# Chapter 5: Dimensionality Reduction

This notebook accompanies **Chapter 5** of the lecture notes.

> A napkin sits on your desk from a coffee run the other day. Someone wrote their phone number on it. The ink has bled, the strokes are jagged, and you cannot quite tell the 3s from the 8s. You could ask them again. Or you could read it the hard way, pixel by pixel, with deep learning.

**Agenda**

🧮 · 📊 · 🧭 · 🪩 · 🗺️ · 🏁

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.manifold import TSNE
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_variance_mask, check_center_data, check_pca_components,
)

## 🧮 784 Dimensions

To read the napkin we need a stand-in for it: **1 000 handwritten digits** from MNIST, each a 28 × 28 image flattened into a 784-feature vector. Ten classes, one per digit, the same alphabet the napkin is written in.

> In 784 dimensions, almost all pairs of points end up at roughly the same Euclidean distance - the contrast between "near" and "far" collapses, and k-NN, clustering, and plotting all suffer. This is the curse of dimensionality. Before we can see, cluster, or classify the napkin, we need to compress it.

In [ ]:
df = pd.read_csv('mnist_digits.csv')
y  = df['label'].values
X  = df.drop('label', axis=1).values.astype(np.float64) / 255.0

print(f'Samples : {X.shape[0]}')
print(f'Features: {X.shape[1]}   (28 x 28 pixels, flattened)')

In [ ]:
_rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 10, figsize=(14, 2.2))
for digit in range(10):
    _idx = _rng.choice(np.where(y == digit)[0])
    axes[digit].imshow(X[_idx].reshape(28, 28), cmap='magma')
    axes[digit].set_title(f'{digit}', fontsize=10, color=_GOLDEN)
    axes[digit].set_xticks([]); axes[digit].set_yticks([])
    for s in axes[digit].spines.values(): s.set_visible(False)
plt.tight_layout()
plt.show()

## 📊 Variance-Based Feature Filtering

The cheapest form of reduction is to **drop features that do not vary**. A pixel that is zero in every image cannot distinguish any two samples - its variance is zero, and removing it loses nothing.

> Corner pixels are black in nearly every digit. Centre pixels carry most of the signal. Should both enter a distance calculation with equal weight?

<details><summary>Thought</summary>

No. A feature with near-zero variance adds noise, not signal, and inflates the dimensionality for nothing. Variance filtering is a preprocessing step, not a replacement for PCA - it can reject flat features but cannot combine them into richer directions.
</details>

Implement `variance_mask`: return a boolean array of length d, True where the feature's variance exceeds the threshold.

Useful operations: `np.var(..., axis=0)`, `>`.

In [ ]:
def variance_mask(X, threshold):
    """Return a boolean mask, True for features with variance greater than threshold."""
    # 1. Compute the variance of each feature (column) with np.var(X, axis=0).
    # 2. Compare against threshold with > (strictly greater, not >=).
    # 3. Return the resulting boolean array of shape (d,).
    # YOUR CODE HERE
    pass


check_variance_mask(variance_mask, X, 0.01)

In [ ]:
_mask = variance_mask(X, 0.01)

if _mask is None:
    print('⬜ Implement variance_mask above first.')
else:
    _mask = np.asarray(_mask, dtype=bool)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.var(X, axis=0).reshape(28, 28), cmap='magma')
    axes[0].set_title('Per-pixel variance', fontsize=10, color=_GOLDEN)
    axes[1].imshow(_mask.reshape(28, 28), cmap='magma')
    axes[1].set_title(f'Kept: {int(np.sum(_mask))}', fontsize=10, color=_ACCENT)
    axes[2].imshow((~_mask).reshape(28, 28), cmap='magma')
    axes[2].set_title(f'Dropped: {int(np.sum(~_mask))}', fontsize=10, color=_TERRA)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
    plt.tight_layout()
    plt.show()

**Observe:**
- The high-variance pixels form a roughly digit-shaped region in the centre of the frame.
- Corners and edges are dropped, they carry no information.
- Variance filtering removes dead features for free, but it cannot discover new directions. That is the job of PCA.

## 🧭 PCA: Centering First

PCA finds the directions of maximum variance in the data. It is linear, global, and cheap - but it depends on a preprocessing step that is easy to skip: **centering**.

> Without centering, the first principal component tends to point toward the mean of the data rather than the direction of spread. Why?

<details><summary>Thought</summary>

Covariance is the expected value of (x - mu)(x - mu) transposed. If the mean is not subtracted, the second moment matrix is inflated by the outer product of the mean, and its top eigenvector chases the offset instead of the variation around it. Centering is what turns "direction of biggest squared values" into "direction of biggest spread".
</details>

Implement `center_data`: subtract the per-feature (column) mean from every sample.

Useful operations: `X.mean(axis=0)`, broadcasting.

In [ ]:
def center_data(X):
    """Subtract the per-feature mean and return the centered array."""
    # 1. Compute one mean per feature (column) with X.mean(axis=0).
    #    This gives a vector of length d, not a scalar.
    # 2. Subtract it from X; numpy broadcasting handles the shape alignment.
    # YOUR CODE HERE
    pass


check_center_data(center_data, X)

In [ ]:
_Xc_preview = center_data(X)

if _Xc_preview is None:
    print('⬜ Implement center_data above first.')
else:
    _mean_img = X.mean(axis=0)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    axes[0].imshow(_mean_img.reshape(28, 28), cmap='magma')
    axes[0].set_title('Mean digit (what centering removes)', fontsize=9, color=_GOLDEN)
    axes[1].imshow(X[0].reshape(28, 28), cmap='magma')
    axes[1].set_title(f'Original sample (label {y[0]})', fontsize=9, color=_GOLDEN)
    _vmax = np.abs(np.asarray(_Xc_preview)[0]).max()
    axes[2].imshow(np.asarray(_Xc_preview)[0].reshape(28, 28),
                   cmap='coolwarm', vmin=-_vmax, vmax=_vmax)
    axes[2].set_title('Centered sample (deviation from mean)', fontsize=9, color=_GOLDEN)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
    plt.tight_layout()
    plt.show()

**Observe:**
- The mean digit is a blurry blob, the average of every class.
- Each centered sample is a deviation from that mean. Blue is darker than average, red is brighter.
- This is the input PCA actually needs. Skipping it is one of the most common PCA bugs.

In [ ]:
# From here on downstream cells need a centered matrix.
# Reference implementation so the notebook can continue if center_data is still a WIP.
X_c = X - X.mean(axis=0)

### Top-k Principal Components

With centered data, PCA is an eigendecomposition of the covariance matrix. The top-k eigenvectors (by eigenvalue) are the directions of largest variance.

Implement `pca_components`: return the top-k eigenvectors of the covariance matrix, sorted by **descending** eigenvalue, as a matrix of shape (k, d), one component per row.

Useful operations: `X_c.T @ X_c`, `np.linalg.eigh`, `np.argsort(...)[::-1]`, transpose.

In [ ]:
def pca_components(X_c, k):
    """Return the top-k eigenvectors of the covariance matrix (shape: (k, d))."""
    # 1. Build the sample covariance matrix from the centered data.
    #    With n samples, the unbiased estimate is X_c.T @ X_c / (n - 1).
    # 2. Get eigenvalues and eigenvectors with np.linalg.eigh.
    #    eigh returns them in ASCENDING order, so you will need to reverse.
    # 3. Sort indices by eigenvalue in descending order: np.argsort(eigvals)[::-1]
    # 4. Pick the first k sorted eigenvectors. eigh returns vectors as COLUMNS,
    #    so you will need to transpose to get shape (k, d), one component per row.
    # YOUR CODE HERE
    pass


check_pca_components(pca_components, X_c, 10)

In [ ]:
_V = pca_components(X_c, 10)

if _V is None:
    print('⬜ Implement pca_components above first.')
else:
    _V = np.asarray(_V)

    fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
    for i in range(6):
        _vmax = np.abs(_V[i]).max()
        axes[i].imshow(_V[i].reshape(28, 28), cmap='coolwarm', vmin=-_vmax, vmax=_vmax)
        axes[i].set_title(f'PC {i+1}', fontsize=9, color=_GOLDEN)
        axes[i].set_xticks([]); axes[i].set_yticks([])
        for s in axes[i].spines.values(): s.set_visible(False)
    plt.tight_layout()
    plt.show()

    _Z = X_c @ _V[:2].T
    fig, ax = plt.subplots(figsize=(8, 6))
    _cmap = plt.get_cmap('tab10', 10)
    for digit in range(10):
        _m = y == digit
        ax.scatter(_Z[_m, 0], _Z[_m, 1], color=_cmap(digit), s=10,
                   linewidths=0, alpha=0.7, label=str(digit))
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title('PCA projection to 2D', fontsize=10, color=_GOLDEN)
    ax.legend(ncol=5, frameon=False, fontsize=8, markerscale=1.8, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- Each principal component is a digit-shaped template, a direction in pixel space along which the dataset spreads the most.
- The 2D PC1-PC2 projection captures coarse structure but collapses many classes on top of each other.
- PCA is **linear**. No rotation of the axes can untangle classes that live on a curved manifold. This is the limit we push against next with t-SNE and UMAP.

## 🪩 t-SNE: Nonlinear Visualization

PCA is linear. On data that lies on a curved manifold, a linear projection cannot untangle it. **t-SNE** takes a different approach: define probabilities over pairs of points (high for close neighbors in the original space), set up a similar distribution in 2D, and iteratively move the 2D points to match the two distributions via gradient descent.

The result is a visualization where **local neighborhoods are preserved** and clusters separate cleanly. The cost: global distances (cluster sizes, empty space, inter-cluster gaps) are **not meaningful**.

> t-SNE's key knob is **perplexity**, which controls the effective number of neighbors each point pays attention to. Too small and clusters fragment; too large and the layout smooths out toward something PCA-like.

t-SNE benefits from a **PCA pre-reduction** when the input is high-dim: run PCA to 30-50 dims first, then t-SNE on that. This speeds up the distance computations and denoises the pairwise distances.

In [ ]:
# PCA pre-reduction (numpy reference, so this runs independent of your pca_components)
_cov_t = X_c.T @ X_c / (X_c.shape[0] - 1)
_e_t, _v_t = np.linalg.eigh(_cov_t)
_V50 = _v_t[:, np.argsort(_e_t)[::-1][:50]].T
_X50 = X_c @ _V50.T

_tsne_results = {}
for _perp in [5, 30, 100]:
    print(f'Running t-SNE with perplexity = {_perp}...')
    _tsne = TSNE(n_components=2, perplexity=_perp, random_state=42,
                 init='pca', learning_rate='auto')
    _tsne_results[_perp] = _tsne.fit_transform(_X50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
_cmap = plt.get_cmap('tab10', 10)
for _ax, _perp in zip(axes, [5, 30, 100]):
    _Z = _tsne_results[_perp]
    for _digit in range(10):
        _m = y == _digit
        _ax.scatter(_Z[_m, 0], _Z[_m, 1], color=_cmap(_digit), s=8,
                    linewidths=0, alpha=0.75, label=str(_digit))
    _ax.set_title(f't-SNE (perplexity = {_perp})', fontsize=10, color=_GOLDEN)
    _ax.set_xticks([]); _ax.set_yticks([])
    tufte_axis(_ax)
axes[-1].legend(ncol=5, frameon=False, fontsize=7, markerscale=1.6,
                labelcolor=_TEXT, bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

**Observe:**
- At **perplexity 5** single classes fragment into multiple puffs, t-SNE only sees tiny neighborhoods.
- At **perplexity 30** the ten classes separate cleanly. This is the sweet spot for 1 000 samples.
- At **perplexity 100** classes blend back together, the local structure that made t-SNE attractive is lost.
- Distances in any of these plots are not calibrated globally. t-SNE is for **visualization**, not downstream inference.

## 🗺️ UMAP

**UMAP** shares the spirit of t-SNE, find a 2D layout where local neighborhoods are preserved, but uses a different mathematical foundation and tends to:

- run faster on larger datasets,
- preserve more **global** structure (relative positions of clusters),
- offer a reusable `.transform` for new samples.

The parameter that plays the role of perplexity is `n_neighbors`.

> `umap-learn` is not available in the JupyterLite sandbox we run in. The embeddings below were **precomputed offline** with default UMAP settings and several `n_neighbors` values.

In [ ]:
_um = pd.read_csv('umap_precomputed.csv')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
_cmap = plt.get_cmap('tab10', 10)
for _ax, _nn in zip(axes, [5, 15, 50]):
    _zx = _um[f'umap_nn{_nn}_x'].values
    _zy = _um[f'umap_nn{_nn}_y'].values
    for _digit in range(10):
        _m = y == _digit
        _ax.scatter(_zx[_m], _zy[_m], color=_cmap(_digit), s=8,
                    linewidths=0, alpha=0.75, label=str(_digit))
    _ax.set_title(f'UMAP (n_neighbors = {_nn})', fontsize=10, color=_GOLDEN)
    _ax.set_xticks([]); _ax.set_yticks([])
    tufte_axis(_ax)
axes[-1].legend(ncol=5, frameon=False, fontsize=7, markerscale=1.6,
                labelcolor=_TEXT, bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

**Observe:**
- All three settings separate the ten classes cleanly.
- Small `n_neighbors` gives tight cluster-shaped blobs (t-SNE-like). Large `n_neighbors` produces a smoother layout where inter-cluster distances become a bit more meaningful.
- Like t-SNE, UMAP is iterative, but unlike t-SNE it exposes a `.transform` for out-of-sample projection.

### 🏁 Recap

**What we did:**
- 🧮 Motivated dim reduction with the curse of dimensionality on 784-dim MNIST.
- 📊 Built a variance-based feature filter as cheap preprocessing.
- 🧭 Derived PCA as an eigendecomposition of the centered covariance matrix.
- 🪩 Explored t-SNE and the effect of perplexity on the layout.
- 🗺️ Compared UMAP embeddings at several `n_neighbors` values.

**Key takeaways:**
- Centering is not optional for PCA. Without it, the top component captures the mean instead of the spread.
- PCA is linear and preserves global geometry. t-SNE and UMAP are nonlinear and preserve local neighborhoods at the cost of global distances. Match the tool to the downstream question, see the lecture notes for the full discussion.
- The napkin is readable now. Whether it was worth the trouble is between you and whoever wrote it.